# Problema 8.1 -- Localizzazione capacitata (costo minimo)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/modellazione-mip/blob/main/notebooks/fam08_1_capacitata.ipynb)

Attivazione aggregata fra la variabile binaria x_l (apri la sede l) e le
variabili continue di flusso y_lc: il legame si dimostra nei due versi
esattamente come nel problema 7.2, ma qui il vincolo di link è anche un
vincolo di capacità (una sola famiglia di vincoli fa entrambe le cose).

Il capitolo completo — modello, dati, risultati e analisi di sensitività — è [sul sito](https://fabiofurini.github.io/modellazione-mip/localizzazione-1/).

## Preparazione

La cella qui sotto installa `gurobipy` e scarica i tre moduli comuni del corso:
`stile.py` (palette), `mip.py` (rilassamento, duale, bound) ed `euristiche.py`
(next-fit, first-fit, best-fit). La licenza inclusa nel pacchetto pip è limitata a **2000
variabili e 2000 vincoli**: le istanze del corso sono piccole e ci stanno tutte
con ampio margine. Per istanze più grandi si attiva la licenza accademica
gratuita da [portal.gurobi.com](https://portal.gurobi.com).

In [ ]:
# Ambiente: il solver e i moduli comuni del corso.
# In locale usa il python/stile.py del repository; su Colab installa e scarica quello che manca.
import importlib.util
import subprocess
import sys
import urllib.request
from pathlib import Path

if importlib.util.find_spec("gurobipy") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "gurobipy", "matplotlib", "pandas", "scipy"], check=True)

for modulo in ('stile', 'mip', 'euristiche'):                     # stile grafico e utilità del corso
    if importlib.util.find_spec(modulo) is None:
        locale = next((p for p in (Path(f"../python/{modulo}.py"), Path(f"python/{modulo}.py"))
                       if p.exists()), None)
        if locale is not None:
            sys.path.insert(0, str(locale.parent.resolve()))   # notebook aperto nel repository
        else:
            urllib.request.urlretrieve(f"https://raw.githubusercontent.com/fabiofurini/modellazione-mip/main/python/{modulo}.py", f"{modulo}.py")   # Colab

In [ ]:
import gurobipy as gp
import pandas as pd
from gurobipy import GRB

from mip import (ammissibile, due_rilassamenti, frazione, nuovo_modello,
                 registra_bound, risolvi, stampa_soluzione, valuta)
from stile import CICLO, intestazione, plt, salva_dati, salva_figura

R = range

# ---------- 1. MODELLO E ISTANZA ----------

intestazione("1. Localizzazione capacitata: dove aprire, quanto spedire")
t1 = [[4, 5, 6], [6, 4, 3]]      # costo di trasporto sede l -> cliente c
u1 = [50, 50]                    # capacita' delle sedi
i1 = [60, 90]                    # costo di apertura
d1 = [8, 25, 27]                 # domanda dei clienti
m, n = 2, 3
salva_dati(pd.DataFrame([{"sede": l + 1, "cliente": c + 1, "t": t1[l][c]}
                         for l in R(m) for c in R(n)]), "loc1_costi")
salva_dati(pd.DataFrame({"sede": R(1, m + 1), "u": u1, "i": i1}), "loc1_sedi")
salva_dati(pd.DataFrame({"cliente": R(1, n + 1), "d": d1}), "loc1_clienti")


def modello_1(t, u, i, d):
    m, n = len(u), len(d)
    mod = nuovo_modello("localizzazione_capacitata")
    x = mod.addVars(m, vtype=GRB.BINARY, name="x")
    y = mod.addVars(m, n, name="y")
    mod.setObjective(gp.quicksum(i[l] * x[l] for l in R(m))
                      + gp.quicksum(t[l][c] * y[l, c] for l in R(m) for c in R(n)), GRB.MINIMIZE)
    mod.addConstrs((u[l] * x[l] - gp.quicksum(y[l, c] for c in R(n)) >= 0 for l in R(m)),
                   name="capacita")
    mod.addConstrs((gp.quicksum(y[l, c] for l in R(m)) == d[c] for c in R(n)), name="domanda")
    return mod, x, y


def duale_1(t, u, i, d):
    """min sum d_c pi_c;  u_l mu_l <= i_l;  -mu_l + pi_c <= t_lc;  mu >= 0, pi libere."""
    m, n = len(u), len(d)
    dl = nuovo_modello("duale_localizzazione")
    mu = dl.addVars(m, name="mu")
    pi = dl.addVars(n, lb=-GRB.INFINITY, name="pi")
    dl.setObjective(gp.quicksum(d[c] * pi[c] for c in R(n)), GRB.MAXIMIZE)
    dl.addConstrs((u[l] * mu[l] <= i[l] for l in R(m)), name="rc_x")
    dl.addConstrs((-mu[l] + pi[c] <= t[l][c] for l in R(m) for c in R(n)), name="rc_y")
    return dl


m1, x1, y1 = modello_1(t1, u1, i1, d1)

# ---------- 2. EURISTICA COSTRUTTIVA (UPPER BOUND) ----------

print("Euristica: si scandiscono le sedi in ordine, riempendo la domanda residua dei clienti")
print("con la capacita' residua di ciascuna sede, senza superare né l'una né l'altra.")


def euristica_1(t, u, i, d):
    m, n = len(u), len(d)
    y, x, rc, rd, passi = {}, [0] * m, list(u), list(d), []
    for l in R(m):
        for c in R(n):
            if rd[c] > 0 and rc[l] > 0:
                q = min(rd[c], rc[l])
                y[(l, c)] = q
                rd[c] -= q
                rc[l] -= q
                passi.append(f"Sede {l + 1}, cliente {c + 1}: si spedisce min(rd={rd[c] + q}, rc={rc[l] + q}) = {q}; "
                             f"rd[{c + 1}] = {rd[c]}, rc[{l + 1}] = {rc[l]}.")
        if rc[l] < u[l]:
            x[l] = 1
            passi.append(f"La sede {l + 1} ha spedito qualcosa (rc = {rc[l]} < u = {u[l]}): si apre, x[{l + 1}] = 1.")
    ok = all(v == 0 for v in rd)
    return x, y, passi, ok


xe, ye, passi, ok = euristica_1(t1, u1, i1, d1)
for i, s in enumerate(passi, 1):
    print(f"  Passo {i}. {s}")
assert ok, "euristica non ammissibile: domanda non soddisfatta"
ub1 = sum(i1[l] * xe[l] for l in R(m)) + sum(t1[l][c] * ye.get((l, c), 0) for l in R(m) for c in R(n))
sol_eur = {f"x[{l}]": xe[l] for l in R(m)}
sol_eur.update({f"y[{l},{c}]": v for (l, c), v in ye.items()})
assert ammissibile(m1, sol_eur)
print(f"  ub = {ub1}")

# ---------- 3. RILASSAMENTO LP E DUALE (LOWER BOUND) ----------

d1_ = duale_1(t1, u1, i1, d1)
mano = {f"mu[{l}]": i1[l] / u1[l] for l in R(m)}
mano.update({f"pi[{c}]": min(t1[l][c] + mano[f"mu[{l}]"] for l in R(m)) for c in R(n)})
lb1, viol = valuta(d1_, mano)
assert viol <= 1e-9, viol
print("Soluzione duale a mano: mu_l = i_l/u_l = " + ", ".join(frazione(i1[l] / u1[l]) for l in R(m))
      + ";  pi_c = min_l (t_lc + mu_l) = " + ", ".join(frazione(mano[f"pi[{c}]"]) for c in R(n))
      + f"  ->  lb = {frazione(lb1)}")
zlp1, zlp1r, _ = due_rilassamenti(m1, d1_)

# ---------- 4. SOLUZIONE OTTIMA DEL MILP ----------

z1 = risolvi(m1)
print("Soluzione ottima del MILP:")
stampa_soluzione(m1, solo_non_nulle=True)
riga = registra_bound("1 localizzazione capacitata", ub1, lb1, zlp1, zlp1r, z1)
salva_dati(pd.DataFrame([riga]), "loc1_bound")

# ---------- 5. DOMANDE DI MODELLAZIONE AGGIUNTIVE ----------

varianti = {}


def variante(nome, mod):
    z = risolvi(mod)
    print(f"  {nome:70s} z = {frazione(z)}")
    return z


# 1a: ogni sede aperta deve spedire almeno 5 litri (lotto minimo / semicontinua)
mod, x, y = modello_1(t1, u1, i1, d1)
mod.addConstrs((gp.quicksum(y[l, c] for c in R(n)) >= 5 * x[l] for l in R(m)), name="lotto_minimo")
varianti["1a"] = variante("1a. Ogni sede aperta spedisce almeno 5 litri (sum_c y_lc >= 5 x_l)", mod)
# 1b: la sede 2 si apre solo se si apre la sede 1
mod, x, y = modello_1(t1, u1, i1, d1)
mod.addConstr(x[1] <= x[0], name="2_solo_se_1")
varianti["1b"] = variante("1b. La sede 2 si apre solo se si apre la sede 1 (x_2 <= x_1)", mod)
salva_dati(pd.DataFrame({"variante": list(varianti), "z": list(varianti.values())}), "loc1_varianti")

# ---------- 6. FIGURE ----------


def barre_flusso(y, m, n, titolo, nome):
    """Per ogni sede, barra impilata dei litri spediti a ciascun cliente."""
    fig, ax = plt.subplots(figsize=(7.2, 3.0))
    for l in R(m):
        inizio = 0
        for c in R(n):
            q = y.get((l, c), 0)
            if q > 0:
                ax.barh(l, q, left=inizio, color=CICLO[c % len(CICLO)], edgecolor="white")
                ax.text(inizio + q / 2, l, f"c{c + 1}", ha="center", va="center", color="white",
                        fontsize=9, fontweight="bold")
                inizio += q
    ax.set_yticks(R(m))
    ax.set_yticklabels([f"sede {l + 1}" for l in R(m)])
    ax.set_xlabel("litri spediti")
    ax.set_title(titolo)
    ax.invert_yaxis()
    salva_figura(fig, nome)


ott_y = {(l, c): y1[l, c].X for l in R(m) for c in R(n) if y1[l, c].X > 1e-6}
barre_flusso(ott_y, m, n, f"Localizzazione capacitata: soluzione ottima (z = {frazione(z1)})", "cap08_capacitata_ottimo")
print("Fine.")

---

Notebook generato da `python/fam08_1_capacitata.py` con `python3 python/genera_notebook.py`:
le modifiche si fanno sullo script, non qui.

Materiale didattico di [Fabio Furini](https://sites.google.com/view/fabiofurini/home-page) — DIAG, Sapienza Università di Roma.
Testi, figure e dati [CC BY 4.0](https://github.com/fabiofurini/modellazione-mip/blob/main/LICENSE),
codice [MIT](https://github.com/fabiofurini/modellazione-mip/blob/main/LICENSE-CODE).